In [ ]:
# ============================================================
# CELL 1 — IMPORTS + LOAD DATABASE
# ============================================================

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.patches import Rectangle

# ============================================================
# PROJECT PATH
# ============================================================

PROJECT_DIR = Path(
    "/home/manish/trading_project/futures_project/fno_stocks_project"
)

DB_PATH = PROJECT_DIR / "database" / "fno_stocks.db"

# ============================================================
# CHECK DATABASE
# ============================================================

print("=" * 60)
print("DATABASE CHECK")
print("=" * 60)

print("Database:")
print(DB_PATH)

print("\nExists:", DB_PATH.exists())

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found:\n{DB_PATH}"
    )

print(
    "Size:",
    f"{DB_PATH.stat().st_size:,} bytes"
)

# ============================================================
# LOAD DATA
# ============================================================

conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query(
    """
    SELECT
        stock,
        symbol,
        contract,
        expiry,
        datetime,
        open,
        high,
        low,
        close,
        volume,
        open_interest
    FROM fno_stocks_data
    ORDER BY stock, expiry, datetime
    """,
    conn
)

conn.close()

# ============================================================
# DATETIME + EXPIRY
# ============================================================

df["datetime"] = pd.to_datetime(
    df["datetime"]
)

df["expiry"] = pd.to_datetime(
    df["expiry"]
).dt.date

# ============================================================
# REMOVE DUPLICATES
# ============================================================

df = (
    df
    .drop_duplicates(
        subset=[
            "stock",
            "symbol",
            "expiry",
            "datetime"
        ]
    )
    .sort_values(
        [
            "stock",
            "expiry",
            "datetime"
        ]
    )
    .reset_index(drop=True)
)

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("F&O STOCK DATABASE LOADED")
print("=" * 60)

print(
    f"Rows      : {len(df):,}"
)

print(
    f"Stocks    : {df['stock'].nunique()}"
)

print(
    f"Contracts : {df['symbol'].nunique()}"
)

print("\nDate Range:")

print(
    df["datetime"].min()
)

print(
    df["datetime"].max()
)

print("=" * 60)

In [ ]:
# ============================================================
# CELL 2 — VALID OI PREPARATION
# ============================================================

# Keep the raw database data unchanged.
# For OI-based analysis, OI <= 0 is treated as invalid.

analysis = df.copy()

# ============================================================
# VALID OI
# ============================================================

analysis["valid_oi"] = (
    analysis["open_interest"]
    .where(
        analysis["open_interest"] > 0
    )
)

# ============================================================
# BASIC OI QUALITY CHECK
# ============================================================

print("=" * 60)
print("OI QUALITY CHECK")
print("=" * 60)

print(
    f"Total rows       : {len(analysis):,}"
)

print(
    f"Valid OI rows    : "
    f"{analysis['valid_oi'].notna().sum():,}"
)

print(
    f"Invalid OI rows  : "
    f"{analysis['valid_oi'].isna().sum():,}"
)

print(
    f"Negative OI rows : "
    f"{(analysis['open_interest'] < 0).sum():,}"
)

print(
    f"Zero OI rows     : "
    f"{(analysis['open_interest'] == 0).sum():,}"
)

print("=" * 60)

In [ ]:
## once load dataset start from here to stock analysis

In [ ]:
# ============================================================
# CELL 3 — STOCK FUTURES ANALYSIS SELECTOR
# ============================================================

available_stocks = sorted(
    analysis["stock"]
    .dropna()
    .unique()
)

print("=" * 60)
print("STOCK FUTURES ANALYSIS")
print("=" * 60)

print(
    f"\nAvailable stocks: "
    f"{len(available_stocks)}"
)

# ============================================================
# 1. STOCK SELECTION
# ============================================================

while True:

    stock_input = input(
        "\nEnter stock symbol "
        "(example: RELIANCE, TCS, INFY): "
    ).strip().upper()

    if stock_input in available_stocks:

        selected_stock = stock_input
        break

    print(
        f"'{stock_input}' not found. "
        "Please enter a valid stock symbol."
    )

print(
    f"\nSelected Stock: "
    f"{selected_stock}"
)

# ============================================================
# 2. ANALYSIS MODE
# ============================================================

print("\nAnalysis Mode:")

print(
    "1. Individual Contract"
)

print(
    "2. Cumulative "
    "(Current + Next + Far)"
)

while True:

    mode_input = input(
        "\nSelect analysis mode (1-2): "
    ).strip()

    if mode_input == "1":

        analysis_mode = "Individual"
        break

    elif mode_input == "2":

        analysis_mode = "Cumulative"
        break

    print(
        "Invalid selection. "
        "Enter 1 or 2."
    )

print(
    f"\nAnalysis Mode: "
    f"{analysis_mode}"
)

# ============================================================
# 3. CONTRACT SELECTION
#    INDIVIDUAL MODE ONLY
# ============================================================

if analysis_mode == "Individual":

    stock_contracts = (
        analysis[
            analysis["stock"] == selected_stock
        ][
            ["symbol", "expiry"]
        ]
        .drop_duplicates()
        .sort_values("expiry")
        .reset_index(drop=True)
    )

    print("\nAvailable contracts:")

    for i, (_, row) in enumerate(
        stock_contracts.iterrows(),
        start=1
    ):

        print(
            f"{i}. "
            f"{row['expiry']} → "
            f"{row['symbol']}"
        )

    while True:

        contract_input = input(
            f"\nSelect contract "
            f"(1-{len(stock_contracts)}): "
        ).strip()

        if contract_input.isdigit():

            contract_choice = int(
                contract_input
            )

            if (
                1 <= contract_choice
                <= len(stock_contracts)
            ):
                break

        print(
            f"Invalid contract. "
            f"Enter 1-{len(stock_contracts)}."
        )

    selected_contract = (
        stock_contracts.iloc[
            contract_choice - 1
        ]
    )

    selected_symbol = (
        selected_contract["symbol"]
    )

    selected_expiry = (
        selected_contract["expiry"]
    )

    print(
        f"\nSelected Contract: "
        f"{selected_symbol}"
    )

# ============================================================
# 4. CUMULATIVE MODE
#    CURRENT + NEXT + FAR
# ============================================================

else:

    stock_contracts = (
        analysis[
            analysis["stock"] == selected_stock
        ][
            ["symbol", "expiry"]
        ]
        .drop_duplicates()
        .sort_values("expiry")
        .reset_index(drop=True)
    )

    cumulative_contracts = (
        stock_contracts
        .head(3)
        .copy()
    )

    print("\nContracts included:")

    for _, row in cumulative_contracts.iterrows():

        print(
            f"  {row['expiry']} → "
            f"{row['symbol']}"
        )

# ============================================================
# 5. TIMEFRAME SELECTION
# ============================================================

timeframes = [
    "5min",
    "15min",
    "30min",
    "1h",
    "4h",
    "1D",
    "1W"
]

print("\nAvailable timeframes:")

for i, timeframe in enumerate(
    timeframes,
    start=1
):

    print(
        f"{i}. {timeframe}"
    )

while True:

    timeframe_input = input(
        "\nEnter timeframe number (1-7): "
    ).strip()

    if timeframe_input.isdigit():

        timeframe_choice = int(
            timeframe_input
        )

        if (
            1 <= timeframe_choice
            <= len(timeframes)
        ):

            selected_timeframe = (
                timeframes[
                    timeframe_choice - 1
                ]
            )

            break

    print(
        "Invalid selection. "
        "Enter a number from 1 to 7."
    )

print(
    f"\nSelected timeframe: "
    f"{selected_timeframe}"
)

print("=" * 60)

In [ ]:
# ============================================================
# CELL 4 — RESAMPLE STOCK FUTURES
# INDIVIDUAL / CUMULATIVE
# ============================================================

# ============================================================
# STOCK DATA
# ============================================================

stock_data = df[
    df["stock"] == selected_stock
].copy()

# ============================================================
# HELPER FUNCTION — RESAMPLE OHLC
# ============================================================

def resample_ohlc(data, timeframe):

    data = (
        data
        .sort_values("datetime")
        .set_index("datetime")
    )

    if timeframe in ["1D", "1W"]:

        result = (
            data
            .resample(timeframe)
            .agg({
                "open": "first",
                "high": "max",
                "low": "min",
                "close": "last"
            })
        )

    else:

        result = (
            data
            .resample(
                timeframe,
                origin="start_day",
                offset="9h15min"
            )
            .agg({
                "open": "first",
                "high": "max",
                "low": "min",
                "close": "last"
            })
        )

    return result


# ============================================================
# HELPER FUNCTION — RESAMPLE VOLUME + OI
# ============================================================

def resample_volume_oi(data, timeframe):

    data = (
        data
        .sort_values("datetime")
        .set_index("datetime")
    )

    if timeframe in ["1D", "1W"]:

        result = (
            data
            .resample(timeframe)
            .agg({
                "volume": "sum",
                "open_interest": "last"
            })
        )

    else:

        result = (
            data
            .resample(
                timeframe,
                origin="start_day",
                offset="9h15min"
            )
            .agg({
                "volume": "sum",
                "open_interest": "last"
            })
        )

    return result


# ============================================================
# INDIVIDUAL CONTRACT
# ============================================================

if analysis_mode == "Individual":

    selected_data = stock_data[
        (stock_data["symbol"] == selected_symbol) &
        (stock_data["expiry"] == selected_expiry)
    ].copy()

    resampled = resample_ohlc(
        selected_data,
        selected_timeframe
    )

    volume_oi = resample_volume_oi(
        selected_data,
        selected_timeframe
    )

    resampled["volume"] = volume_oi["volume"]
    resampled["open_interest"] = volume_oi[
        "open_interest"
    ]


# ============================================================
# CUMULATIVE — CURRENT + NEXT + FAR
# ============================================================

else:

    # --------------------------------------------------------
    # Current contract
    # --------------------------------------------------------

    current_contract = (
        cumulative_contracts
        .iloc[0]
    )

    current_symbol = current_contract["symbol"]
    current_expiry = current_contract["expiry"]

    current_data = stock_data[
        (stock_data["symbol"] == current_symbol) &
        (stock_data["expiry"] == current_expiry)
    ].copy()

    # --------------------------------------------------------
    # Current contract → PRICE
    # --------------------------------------------------------

    resampled = resample_ohlc(
        current_data,
        selected_timeframe
    )

    # --------------------------------------------------------
    # Current + Next + Far
    # → Volume + OI
    # --------------------------------------------------------

    contract_results = []

    for _, contract in cumulative_contracts.iterrows():

        symbol = contract["symbol"]
        expiry = contract["expiry"]

        contract_data = stock_data[
            (stock_data["symbol"] == symbol) &
            (stock_data["expiry"] == expiry)
        ].copy()

        contract_resampled = resample_volume_oi(
            contract_data,
            selected_timeframe
        )

        contract_results.append(
            contract_resampled
        )

    # --------------------------------------------------------
    # COMBINE CONTRACTS
    # --------------------------------------------------------

    cumulative_data = pd.concat(
        contract_results,
        axis=1,
        keys=range(
            len(contract_results)
        )
    )

    # --------------------------------------------------------
    # SUM VOLUME
    # --------------------------------------------------------

    cumulative_volume = (
        cumulative_data
        .xs(
            "volume",
            axis=1,
            level=1
        )
        .sum(
            axis=1,
            min_count=1
        )
    )

    # --------------------------------------------------------
    # SUM OI
    # --------------------------------------------------------

    cumulative_oi = (
        cumulative_data
        .xs(
            "open_interest",
            axis=1,
            level=1
        )
        .sum(
            axis=1,
            min_count=1
        )
    )

    # --------------------------------------------------------
    # ADD TO PRICE DATA
    # --------------------------------------------------------

    resampled["volume"] = cumulative_volume
    resampled["open_interest"] = cumulative_oi


# ============================================================
# REMOVE EMPTY PRICE CANDLES
# ============================================================

resampled = resampled.dropna(
    subset=[
        "open",
        "high",
        "low",
        "close"
    ]
)


# ============================================================
# INTRADAY SESSION FILTER
# ============================================================

if selected_timeframe not in ["1D", "1W"]:

    resampled = resampled[
        (
            resampled.index.time >=
            pd.Timestamp("09:15").time()
        )
        &
        (
            resampled.index.time <=
            pd.Timestamp("15:30").time()
        )
    ]


# ============================================================
# RESET INDEX
# ============================================================

resampled = resampled.reset_index()


# ============================================================
# DISPLAY
# ============================================================

print("=" * 60)
print("RESAMPLED STOCK FUTURES DATA")
print("=" * 60)

print(
    f"Stock     : {selected_stock}"
)

print(
    f"Mode      : {analysis_mode}"
)

print(
    f"Timeframe : {selected_timeframe}"
)

if analysis_mode == "Individual":

    print(
        f"Contract  : {selected_symbol}"
    )

else:

    print(
        f"Price     : {current_symbol}"
    )

    print(
        "OI/Volume : Current + Next + Far"
    )

print()
print(
    f"Candles   : {len(resampled)}"
)

print(
    "Start     :",
    resampled["datetime"].min()
)

print(
    "End       :",
    resampled["datetime"].max()
)

print()
print("Recent data:")

print(
    resampled[
        [
            "datetime",
            "close",
            "volume",
            "open_interest"
        ]
    ].tail(10)
)

In [ ]:
# ============================================================
# CELL 5 — TIMEFRAME F&O ANALYSIS
# INDIVIDUAL / CUMULATIVE
# ============================================================

tf_analysis = resampled.copy()

# ============================================================
# PREVIOUS CLOSE
# ============================================================

tf_analysis["prev_close"] = (
    tf_analysis["close"].shift(1)
)

# ============================================================
# VALID OI
# ============================================================

tf_analysis["valid_oi"] = (
    tf_analysis["open_interest"]
    .where(
        tf_analysis["open_interest"] > 0
    )
)

tf_analysis["prev_oi"] = (
    tf_analysis["valid_oi"].shift(1)
)

# ============================================================
# PRICE CHANGE
# ============================================================

tf_analysis["price_change"] = (
    tf_analysis["close"]
    - tf_analysis["prev_close"]
)

tf_analysis["price_change_pct"] = (
    tf_analysis["price_change"]
    / tf_analysis["prev_close"]
    * 100
)

# ============================================================
# OI CHANGE
# ============================================================

tf_analysis["oi_change"] = (
    tf_analysis["valid_oi"]
    - tf_analysis["prev_oi"]
)

tf_analysis["oi_change_pct"] = (
    tf_analysis["oi_change"]
    / tf_analysis["prev_oi"]
    * 100
)

# ============================================================
# POSITION CLASSIFICATION
# ============================================================

def classify_position(row):

    price_change = row["price_change"]
    oi_change = row["oi_change"]

    # No valid OI comparison
    if pd.isna(oi_change):
        return "No OI Data"

    # Price UP + OI UP
    if price_change > 0 and oi_change > 0:
        return "Long Buildup"

    # Price DOWN + OI UP
    elif price_change < 0 and oi_change > 0:
        return "Short Buildup"

    # Price UP + OI DOWN
    elif price_change > 0 and oi_change < 0:
        return "Short Covering"

    # Price DOWN + OI DOWN
    elif price_change < 0 and oi_change < 0:
        return "Long Unwinding"

    # Price unchanged and/or OI unchanged
    else:
        return "Neutral"


tf_analysis["position"] = (
    tf_analysis.apply(
        classify_position,
        axis=1
    )
)

# ============================================================
# SUMMARY
# ============================================================

print("=" * 60)
print("F&O POSITION ANALYSIS")
print("=" * 60)

print(
    f"Stock     : {selected_stock}"
)

print(
    f"Mode      : {analysis_mode}"
)

print(
    f"Timeframe : {selected_timeframe}"
)

if analysis_mode == "Individual":

    print(
        f"Contract  : {selected_symbol}"
    )

else:

    print(
        "Contracts : Current + Next + Far"
    )

    print(
        f"Price     : {current_symbol}"
    )

print()
print("Position summary:")
print(
    tf_analysis["position"]
    .value_counts()
)

print()
print("Recent analysis:")

print(
    tf_analysis[
        [
            "datetime",
            "close",
            "volume",
            "open_interest",
            "price_change_pct",
            "oi_change_pct",
            "position"
        ]
    ].tail(15)
)

In [ ]:
# ============================================================
# CELL 6 — FINAL STOCK FUTURES DASHBOARD
# ============================================================

# ============================================================
# ADD ANALYTICS
# ============================================================

tf_analysis["price_return_pct"] = (
    tf_analysis["close"]
    .pct_change()
    * 100
)

tf_analysis["rolling_std"] = (
    tf_analysis["price_return_pct"]
    .rolling(10)
    .std()
)

# ============================================================
# FIGURE
# ============================================================

fig, axes = plt.subplots(
    6,
    1,
    figsize=(18, 20),
    sharex=True,
    gridspec_kw={
        "height_ratios": [
            4,
            1.3,
            1.3,
            1.2,
            1.2,
            1.2
        ]
    }
)

x = np.arange(
    len(tf_analysis)
)

# ============================================================
# 1. PRICE / CANDLESTICK
# ============================================================

ax = axes[0]

candle_width = 0.65

for i, row in tf_analysis.iterrows():

    # --------------------------------------------------------
    # WICK
    # --------------------------------------------------------

    ax.vlines(
        i,
        row["low"],
        row["high"],
        linewidth=1
    )

    # --------------------------------------------------------
    # BODY
    # --------------------------------------------------------

    body_low = min(
        row["open"],
        row["close"]
    )

    body_height = abs(
        row["close"]
        - row["open"]
    )

    if body_height == 0:

        body_height = 0.01

    rect = Rectangle(
        (
            i - candle_width / 2,
            body_low
        ),
        candle_width,
        body_height,
        fill=False,
        linewidth=1
    )

    ax.add_patch(rect)

# ============================================================
# POSITION DOTS
# ============================================================

position_colors = {
    "Long Buildup": "green",
    "Short Buildup": "red",
    "Short Covering": "blue",
    "Long Unwinding": "orange"
}

for position, color in position_colors.items():

    mask = (
        tf_analysis["position"]
        == position
    )

    ax.scatter(
        x[mask],
        tf_analysis.loc[
            mask,
            "close"
        ],
        s=60,
        color=color,
        label=position,
        zorder=5
    )

# ============================================================
# TITLE
# ============================================================

if analysis_mode == "Cumulative":

    title = (
        f"{selected_stock} Futures — "
        f"CUMULATIVE — "
        f"{selected_timeframe}"
    )

else:

    title = (
        f"{selected_stock} Futures — "
        f"{selected_symbol} — "
        f"{selected_timeframe}"
    )

ax.set_title(
    title,
    fontsize=16,
    fontweight="bold"
)

ax.set_ylabel("Price")

ax.grid(
    True,
    alpha=0.25
)

ax.legend(
    loc="upper left",
    ncol=4
)

# ============================================================
# 2. VOLUME
# ============================================================

axes[1].bar(
    x,
    tf_analysis["volume"],
    width=0.8
)

axes[1].set_ylabel(
    "Volume"
)

axes[1].grid(
    True,
    alpha=0.25
)

# ============================================================
# 3. OPEN INTEREST
# ============================================================

axes[2].plot(
    x,
    tf_analysis["open_interest"],
    linewidth=1.5
)

if analysis_mode == "Cumulative":

    axes[2].set_ylabel(
        "Cumulative OI"
    )

else:

    axes[2].set_ylabel(
        "Open Interest"
    )

axes[2].grid(
    True,
    alpha=0.25
)

# ============================================================
# 4. OI CHANGE %
# ============================================================

axes[3].plot(
    x,
    tf_analysis["oi_change_pct"],
    linewidth=1.2
)

axes[3].axhline(
    0,
    linewidth=1
)

axes[3].set_ylabel(
    "OI Change %"
)

axes[3].grid(
    True,
    alpha=0.25
)

# ============================================================
# 5. PRICE RETURN %
# ============================================================

axes[4].plot(
    x,
    tf_analysis["price_return_pct"],
    linewidth=1.2
)

axes[4].axhline(
    0,
    linewidth=1
)

axes[4].set_ylabel(
    "Price Return %"
)

axes[4].grid(
    True,
    alpha=0.25
)

# ============================================================
# 6. ROLLING STANDARD DEVIATION
# ============================================================

axes[5].plot(
    x,
    tf_analysis["rolling_std"],
    linewidth=1.2
)

axes[5].set_ylabel(
    "Rolling Std Dev"
)

axes[5].set_xlabel(
    "Date"
)

axes[5].grid(
    True,
    alpha=0.25
)

# ============================================================
# X-AXIS
# ============================================================

step = max(
    1,
    len(tf_analysis) // 12
)

tick_positions = x[::step]

tick_labels = (
    tf_analysis["datetime"]
    .dt.strftime("%d-%b")
    .iloc[::step]
)

axes[5].set_xticks(
    tick_positions
)

axes[5].set_xticklabels(
    tick_labels,
    rotation=45,
    ha="right"
)

# ============================================================
# FINAL LAYOUT
# ============================================================

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CELL 7 — LATEST F&O STATUS SUMMARY
# ============================================================

latest = tf_analysis.iloc[-1]

position = latest["position"]

# ============================================================
# POSITION SYMBOLS
# ============================================================

position_symbol = {
    "Long Buildup": "🟢",
    "Short Buildup": "🔴",
    "Short Covering": "🔵",
    "Long Unwinding": "🟠",
    "Neutral": "⚪",
    "No OI Data": "⚪"
}

# ============================================================
# HEADER
# ============================================================

print()
print("=" * 60)

if analysis_mode == "Cumulative":

    print(
        f"{selected_stock} — "
        f"CUMULATIVE FUTURES ANALYSIS"
    )

else:

    print(
        f"{selected_stock} — "
        f"INDIVIDUAL FUTURES ANALYSIS"
    )

print("=" * 60)

# ============================================================
# LATEST DATA
# ============================================================

print()

print(
    f"Latest Date       : "
    f"{latest['datetime'].strftime('%d-%b-%Y')}"
)

print(
    f"Current Price     : "
    f"{latest['close']:.2f}"
)

print(
    f"Volume            : "
    f"{latest['volume']:,.0f}"
)

if analysis_mode == "Cumulative":

    print(
        f"Cumulative OI    : "
        f"{latest['open_interest']:,.0f}"
    )

else:

    print(
        f"Open Interest     : "
        f"{latest['open_interest']:,.0f}"
    )

print(
    f"OI Change         : "
    f"{latest['oi_change_pct']:.2f}%"
)

print(
    f"Price Return      : "
    f"{latest['price_return_pct']:.2f}%"
)

print(
    f"Latest Position   : "
    f"{position_symbol.get(position, '')} "
    f"{position}"
)

# ============================================================
# SIGNAL COUNTS
# ============================================================

print()
print("-" * 60)
print("SIGNAL COUNTS")
print("-" * 60)

signal_counts = (
    tf_analysis["position"]
    .value_counts()
)

for signal in [
    "Long Buildup",
    "Short Buildup",
    "Short Covering",
    "Long Unwinding",
    "Neutral",
    "No OI Data"
]:

    count = signal_counts.get(
        signal,
        0
    )

    if count > 0:

        print(
            f"{position_symbol.get(signal, '')} "
            f"{signal:<18} : {count}"
        )

print("=" * 60)

In [ ]:
# ============================================================
# CELL 8 — NEXT-PERIOD EMPIRICAL PROBABILITY
# ============================================================

prob_data = tf_analysis.copy()

# ============================================================
# TIMEFRAME-BASED MOVE THRESHOLD
# ============================================================

move_thresholds = {
    "5min": 0.15,
    "15min": 0.25,
    "30min": 0.35,
    "1h": 0.50,
    "4h": 0.75,
    "1D": 0.50,
    "1W": 1.00
}

threshold = move_thresholds[
    selected_timeframe
]

# ============================================================
# NEXT PERIOD RETURN
# ============================================================

prob_data["next_return_pct"] = (
    prob_data["close"]
    .shift(-1)
    .div(
        prob_data["close"]
    )
    .sub(1)
    * 100
)

# ============================================================
# CLASSIFY NEXT MOVE
# ============================================================

def classify_next_move(return_pct):

    if pd.isna(return_pct):
        return np.nan

    if return_pct > threshold:

        return "Upside"

    elif return_pct < -threshold:

        return "Downside"

    else:

        return "Sideways"


prob_data["next_move"] = (
    prob_data["next_return_pct"]
    .apply(classify_next_move)
)

# ============================================================
# REMOVE LAST ROW
# ============================================================

historical_moves = (
    prob_data
    .dropna(
        subset=["next_move"]
    )
    .copy()
)

# ============================================================
# OVERALL HISTORICAL PROBABILITY
# ============================================================

probability = (
    historical_moves["next_move"]
    .value_counts(
        normalize=True
    )
    * 100
)

upside_probability = probability.get(
    "Upside",
    0
)

sideways_probability = probability.get(
    "Sideways",
    0
)

downside_probability = probability.get(
    "Downside",
    0
)

# ============================================================
# CURRENT POSITION CONDITIONAL PROBABILITY
# ============================================================

current_position = (
    tf_analysis.iloc[-1]["position"]
)

conditional_data = (
    historical_moves[
        historical_moves["position"]
        == current_position
    ]
    .copy()
)

conditional_probability = (
    conditional_data["next_move"]
    .value_counts(
        normalize=True
    )
    * 100
)

conditional_upside = (
    conditional_probability.get(
        "Upside",
        0
    )
)

conditional_sideways = (
    conditional_probability.get(
        "Sideways",
        0
    )
)

conditional_downside = (
    conditional_probability.get(
        "Downside",
        0
    )
)

# ============================================================
# DISPLAY
# ============================================================

print("=" * 60)
print("NEXT-PERIOD EMPIRICAL PROBABILITY")
print("=" * 60)

print(
    f"Stock     : {selected_stock}"
)

print(
    f"Mode      : {analysis_mode}"
)

print(
    f"Timeframe : {selected_timeframe}"
)

print(
    f"Move threshold : ±{threshold:.2f}%"
)

# ============================================================
# OVERALL
# ============================================================

print()
print("OVERALL HISTORICAL PROBABILITY")
print("-" * 60)

print(
    f"Upside       : "
    f"{upside_probability:.1f}%"
)

print(
    f"Sideways     : "
    f"{sideways_probability:.1f}%"
)

print(
    f"Downside     : "
    f"{downside_probability:.1f}%"
)

print(
    f"Observations : "
    f"{len(historical_moves)}"
)

# ============================================================
# CONDITIONAL
# ============================================================

print()
print(
    f"CURRENT STATE: "
    f"{current_position}"
)

print("-" * 60)

print(
    f"Upside       : "
    f"{conditional_upside:.1f}%"
)

print(
    f"Sideways     : "
    f"{conditional_sideways:.1f}%"
)

print(
    f"Downside     : "
    f"{conditional_downside:.1f}%"
)

print(
    f"Matching historical cases: "
    f"{len(conditional_data)}"
)

print("=" * 60)

In [ ]:
# ============================================================
# CELL 9 — FINAL ANALYSIS SUMMARY
# ============================================================

latest = tf_analysis.iloc[-1]

current_position = latest["position"]

# ============================================================
# SIGNAL DESCRIPTION
# ============================================================

position_description = {
    "Long Buildup":
        "Price is rising while OI is increasing. "
        "This indicates fresh long participation.",

    "Short Buildup":
        "Price is falling while OI is increasing. "
        "This indicates fresh short participation.",

    "Short Covering":
        "Price is rising while OI is decreasing. "
        "This indicates short positions are being reduced.",

    "Long Unwinding":
        "Price is falling while OI is decreasing. "
        "This indicates long positions are being reduced.",

    "Neutral":
        "Price and OI did not show a directional combination.",

    "No OI Data":
        "There is insufficient valid OI data for classification."
}

# ============================================================
# HISTORICAL TENDENCY
# ============================================================

historical_cases = len(conditional_data)

if historical_cases > 0:

    probabilities = {
        "Upside": conditional_upside,
        "Sideways": conditional_sideways,
        "Downside": conditional_downside
    }

    historical_tendency = max(
        probabilities,
        key=probabilities.get
    )

else:

    historical_tendency = "Insufficient Data"

# ============================================================
# DISPLAY
# ============================================================

print()
print("=" * 70)
print("FINAL STOCK FUTURES ANALYSIS SUMMARY")
print("=" * 70)

print()
print(f"Stock          : {selected_stock}")
print(f"Analysis Mode  : {analysis_mode}")
print(f"Timeframe      : {selected_timeframe}")

if analysis_mode == "Individual":

    print(f"Contract       : {selected_symbol}")

else:

    print("Contracts      : Current + Next + Far")
    print(f"Price Contract : {current_symbol}")

print()
print("-" * 70)
print("LATEST MARKET STATE")
print("-" * 70)

print(
    f"Date           : "
    f"{latest['datetime'].strftime('%d-%b-%Y')}"
)

print(
    f"Price          : "
    f"{latest['close']:.2f}"
)

print(
    f"Price Return   : "
    f"{latest['price_return_pct']:.2f}%"
)

print(
    f"OI Change      : "
    f"{latest['oi_change_pct']:.2f}%"
)

print(
    f"Position       : "
    f"{current_position}"
)

print()
print(
    "Interpretation:"
)

print(
    position_description[current_position]
)

print()
print("-" * 70)
print("HISTORICAL NEXT-PERIOD TENDENCY")
print("-" * 70)

print(
    f"Move Threshold : "
    f"±{threshold:.2f}%"
)

print(
    f"Historical Cases: "
    f"{historical_cases}"
)

if historical_cases > 0:

    print()
    print(
        f"Upside        : "
        f"{conditional_upside:.1f}%"
    )

    print(
        f"Sideways      : "
        f"{conditional_sideways:.1f}%"
    )

    print(
        f"Downside      : "
        f"{conditional_downside:.1f}%"
    )

    print()
    print(
        f"Most Frequent Historical Outcome: "
        f"{historical_tendency}"
    )

else:

    print(
        "Not enough matching historical cases."
    )

print()
print("-" * 70)
print(
    "Note: Historical probabilities describe past "
    "observations only and are not guaranteed predictions."
)
print("=" * 70)

In [ ]:
pwd